# Sweep 16b Ablation — MIMIC-IV SOTA Cohort (~9K patients)

**Cohort**: Strict ICD-9 admission-level filter (~9K patients, comparable to published papers)

8 contexts × 3 DDI alphas × 5 seeds = **120 runs**

| Cell | Content | Est. time |
|------|---------|----------|
| 1 | Install | 1 min |
| 2 | Kaggle setup | 2 min |
| 3 | Config + helpers | — |
| 4 | Smoke test | 5 min |
| 5-12 | 8 context run cells (3 alphas each) | ~4.5h each |
| 13 | Run log | — |
| 14 | Results grid | — |
| 15-22 | Per-context export | — |
| 23 | Export all | — |

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    # Processed files — anchor on cohort_mimic4_sota.pkl
    for find_file in ["cohort_mimic4_sota.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings_mimic4_sota.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings_mimic4_sota.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic4_sota.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic4_sota.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic4_sota.pkl not found")

    # Patch 1: registry.py
    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch 2: drug_gnn.py
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched")

    # Patch 3: dcma_decoder.py
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

CHAMPION_MODEL = {
    "hidden_dim":         128,
    "ar_max_seq_len":     20,
    "note_proj_dim":      64,
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type":   "gcn",
    "hgt_layers":         2,
    "encoder_type":       "imdr_infused",
    "predictor_type":     "heidr",
}
CHAMPION_TRAINING = {
    "bce_weight":           0.3,
    "soft_jaccard_weight":  1.5,
    "margin_weight":        0.05,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
    "--mimic_version 4 --mimic4_sota "
    "--processed_dir data/processed "
    "--embeddings_dir data/embeddings "
)
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"

CONTEXTS = {
    "naked":      {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 0},
                   "flags":    "",
                   "label":    "[1] naked         (no copy, no notes, no labs)"},
    "copy":       {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"note_method": "none", "lab_dim": 0},
                   "flags":    "",
                   "label":    "[2] + copy"},
    "notes":      {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 0},
                   "flags":    "",
                   "label":    "[3] + notes               (no copy, no labs)"},
    "labs":       {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[4] + labs                (no copy, no notes)"},
    "notes_copy": {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"lab_dim": 0},
                   "flags":    "",
                   "label":    "[5] + notes + copy        (no labs)"},
    "labs_copy":  {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"note_method": "none", "lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[6] + labs + copy         (no notes)"},
    "notes_labs": {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[7] + notes + labs        (no copy)"},
    "full":       {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[8] + notes + labs + copy  CHAMPION"},
}

SEEDS       = [42, 123, 456, 789, 1024]
DDI_ALPHAS  = [0.0, 0.2, 0.5]
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
results_log = []

def alpha_tag(a): return str(a).replace(".", "p")

def run_context(ctx_key, ddi_alpha=0.0):
    import torch
    ctx  = CONTEXTS[ctx_key]
    atag = alpha_tag(ddi_alpha)
    cfgp = f"s16bm4s_{ctx_key}_alpha{atag}.yaml"
    make_config(cfgp,
                model_overrides={**CHAMPION_MODEL, **ctx["model_ov"]},
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=ctx["prep_ov"])
    print(f"\n  -- {ctx['label']}  |  ddi_alpha={ddi_alpha} --")
    for seed in SEEDS:
        run_name = f"{ctx_key}_mimic4sota_alpha{atag}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        if list(run_dir.glob("result_*.json")):
            print(f"    [SKIP] {run_name}")
            results_log.append(f"SKIP: {run_name}")
            continue
        log_path = run_dir / "training_log.txt"
        cmd = (f"python -u src/train.py --config {cfgp} "
               f"{BACKBONE_FLAGS} {ctx['flags']} "
               f"--ddi_alpha {ddi_alpha} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"    --- {run_name} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Ablation config ready. {len(CONTEXTS)} contexts x {len(DDI_ALPHAS)} alphas x {len(SEEDS)} seeds = {len(CONTEXTS)*len(DDI_ALPHAS)*len(SEEDS)} total runs")
print("Cohort: MIMIC-IV SOTA (~9K patients, strict ICD-9 admission-level)")


In [ ]:
# Smoke test: 1 epoch x 3 contexts x alpha=0.0
Path("smoke_s16bm4s").mkdir(exist_ok=True)
smoke_cases = ["naked", "notes_labs", "full"]
smoke_results = []
for ctx_name in smoke_cases:
    ctx = CONTEXTS[ctx_name]
    cfg_path = f"s16bm4s_smoke_{ctx_name}.yaml"
    make_config(cfg_path,
                model_overrides={**CHAMPION_MODEL, **ctx["model_ov"]},
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=ctx["prep_ov"],
                smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {ctx['flags']} "
           f"--ddi_alpha 0.0 "
           f"--seed 42 --results_dir smoke_s16bm4s/{ctx_name}")
    print(f"SMOKE [{ctx_name}]: running...")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = proc.returncode == 0
    smoke_results.append((ctx_name, "PASS" if ok else "FAIL"))
    for ln in (proc.stdout + proc.stderr).strip().split("\n")[-4:]: print(" ", ln)
    print(f"  -> {'PASS' if ok else 'FAIL'}")

print("\nSmoke summary:")
for ctx, status in smoke_results:
    print(f"  {ctx}: {status}")
if any(s == "FAIL" for _, s in smoke_results):
    raise RuntimeError("Smoke test failed — fix before running full ablation")


In [ ]:
# Cell 5 — [1] naked
for alpha in DDI_ALPHAS:
    run_context("naked", ddi_alpha=alpha)


In [ ]:
# Cell 6 — [2] +copy
for alpha in DDI_ALPHAS:
    run_context("copy", ddi_alpha=alpha)


In [ ]:
# Cell 7 — [3] +notes
for alpha in DDI_ALPHAS:
    run_context("notes", ddi_alpha=alpha)


In [ ]:
# Cell 8 — [4] +labs
for alpha in DDI_ALPHAS:
    run_context("labs", ddi_alpha=alpha)


In [ ]:
# Cell 9 — [5] +notes+copy
for alpha in DDI_ALPHAS:
    run_context("notes_copy", ddi_alpha=alpha)


In [ ]:
# Cell 10 — [6] +labs+copy
for alpha in DDI_ALPHAS:
    run_context("labs_copy", ddi_alpha=alpha)


In [ ]:
# Cell 11 — [7] +notes+labs
for alpha in DDI_ALPHAS:
    run_context("notes_labs", ddi_alpha=alpha)


In [ ]:
# Cell 12 — [8] full champion
for alpha in DDI_ALPHAS:
    run_context("full", ddi_alpha=alpha)


In [ ]:
print(f"\nRun log ({len(results_log)} entries):")
for e in results_log: print(" ", e)


In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")

def alpha_tag(a): return str(a).replace(".", "p")

CONTEXTS_ORDER = ["naked","copy","notes","labs","notes_copy","labs_copy","notes_labs","full"]
LABELS = {
    "naked":      "[1] naked",
    "copy":       "[2] +copy",
    "notes":      "[3] +notes",
    "labs":       "[4] +labs",
    "notes_copy": "[5] +notes+copy",
    "labs_copy":  "[6] +labs+copy",
    "notes_labs": "[7] +notes+labs",
    "full":       "[8] CHAMPION (notes+labs+copy)",
}
DDI_ALPHAS = [0.0, 0.2, 0.5]
SEEDS = [42, 123, 456, 789, 1024]

print("=" * 85)
print("  MIRROR Ablation — MIMIC-IV SOTA (~9K, strict ICD-9) — 8 contexts x 3 alphas")
print("=" * 85)
print(f"{'Context':<24}", end="")
for alpha in DDI_ALPHAS:
    print(f"  alpha={alpha:.1f} Jac  DDI ", end="")
print()
print("-" * 85)

for ctx in CONTEXTS_ORDER:
    print(f"  {LABELS[ctx]:<22}", end="")
    for alpha in DDI_ALPHAS:
        atag = alpha_tag(alpha)
        jacs, ddis = [], []
        for seed in SEEDS:
            rn  = f"{ctx}_mimic4sota_alpha{atag}_seed{seed}"
            rd  = reports_dir / rn
            fs  = list(rd.glob("result_*.json")) if rd.exists() else []
            if not fs: continue
            r   = json.loads(fs[0].read_text())
            tm  = r.get("test_metrics", {})
            jacs.append(tm.get("jaccard", float("nan")))
            ddis.append(tm.get("ddi_rate", float("nan")))
        if jacs:
            jm = np.nanmean(jacs); dm = np.nanmean(ddis)
            ddi_flag = "✓" if dm < 0.06 else " "
            print(f"  {jm:.4f}  {dm:.4f}{ddi_flag}", end="")
        else:
            print(f"  —       —      ", end="")
    print()


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_naked_results"
pattern = f"naked_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for naked yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_copy_results"
pattern = f"copy_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for copy yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_notes_results"
pattern = f"notes_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for notes yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_labs_results"
pattern = f"labs_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for labs yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_notes_copy_results"
pattern = f"notes_copy_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for notes_copy yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_labs_copy_results"
pattern = f"labs_copy_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for labs_copy yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_notes_labs_results"
pattern = f"notes_labs_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for notes_labs yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_name = "sweep_16b_mimic4sota_full_results"
pattern = f"full_mimic4sota_*"
import tempfile, os
tmp = Path(tempfile.mkdtemp())
for d in reports_dir.glob(pattern):
    shutil.copytree(d, tmp / d.name)
if list(tmp.iterdir()):
    shutil.make_archive(zip_name, "zip", tmp)
    print(f"Exported: {zip_name}.zip  ({os.path.getsize(zip_name+'.zip')/1e6:.1f} MB)")
else:
    print("No results for full yet.")
shutil.rmtree(tmp)


In [ ]:
import shutil, os
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4sota")
zip_path = "sweep_16b_ablation_mimic4sota_all_results.zip"
if reports_dir.exists():
    shutil.make_archive(zip_path.replace(".zip",""), "zip", reports_dir)
    print(f"Exported: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
else:
    print("No results directory yet.")
